# 🕷️ Orphan Page Checker
> Temukan halaman yang terdaftar di sitemap tapi **tidak bisa dijangkau** dari homepage.

**Urutan penggunaan:**
1. Jalankan `sitemap_crawler.ipynb` terlebih dahulu → download CSV sitemap
2. Jalankan **Cell 1** di bawah → input homepage + upload CSV sitemap → tunggu selesai ☕
3. Jalankan **Cell 2** → hasil orphan pages otomatis ter-download

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Setup Input + Website Crawler                 ║
# ╚══════════════════════════════════════════════════════════╝

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import io
from datetime import datetime
from urllib.parse import urlparse, urljoin, urldefrag
from google.colab import files as colab_files

import subprocess
subprocess.run(["pip", "install", "rich", "-q"])

from rich.console import Console
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()

# ── Banner ────────────────────────────────────────────────────────────────────
console.print(Panel.fit(
    "[bold magenta]🔍 ORPHAN PAGE CHECKER[/bold magenta]\n"
    "[dim]Sitemap Crawler + Internal Link Crawler + Orphan Detector[/dim]",
    border_style="magenta"
))

# ── Input semua di awal ───────────────────────────────────────────────────────
console.print("\n[bold cyan]━━━ INPUT SETUP ━━━[/bold cyan]")

HOMEPAGE = console.input(
    "\n[bold yellow]🌐 [1/2] Masukkan URL Homepage[/bold yellow] "
    "(contoh: https://www.balitouristic.com): "
).strip().rstrip("/")

console.print("\n[bold yellow]📂 [2/2] Upload CSV hasil sitemap_crawler.ipynb:[/bold yellow]")
uploaded         = colab_files.upload()
sitemap_csv_name = list(uploaded.keys())[0]
df_sitemap       = pd.read_csv(io.BytesIO(uploaded[sitemap_csv_name]))

# ── Derive domain info ────────────────────────────────────────────────────────
def get_domain_name(url: str) -> str:
    try:
        parsed = urlparse(url)
        domain = parsed.netloc or parsed.path
        domain = re.sub(r'^www\.', '', domain)
        return re.sub(r'[^\w]', '_', domain)
    except Exception:
        return "site"

def get_base_domain(url: str) -> str:
    parsed = urlparse(url)
    return parsed.scheme + "://" + parsed.netloc

DOMAIN_NAME   = get_domain_name(HOMEPAGE)
BASE_DOMAIN   = get_base_domain(HOMEPAGE)
OUTPUT_CRAWL  = f"{DOMAIN_NAME}_crawled_urls.csv"
OUTPUT_ORPHAN = f"{DOMAIN_NAME}_orphan_pages.csv"
START_TIME    = datetime.now()

console.print(f"\n[bold green]✔ Semua input diterima! Proses akan jalan otomatis sampai selesai.[/bold green]")
console.print(Panel(
    f"  [dim]Domain[/dim]         : [bold]{DOMAIN_NAME}[/bold]\n"
    f"  [dim]Base URL[/dim]       : [bold cyan]{BASE_DOMAIN}[/bold cyan]\n"
    f"  [dim]Sitemap CSV[/dim]    : [cyan]{sitemap_csv_name}[/cyan]  ([bold]{len(df_sitemap)}[/bold] entri)\n"
    f"  [dim]Output crawl[/dim]   : [cyan]{OUTPUT_CRAWL}[/cyan]\n"
    f"  [dim]Output orphan[/dim]  : [cyan]{OUTPUT_ORPHAN}[/cyan]\n"
    f"  [dim]Mulai[/dim]          : [dim]{START_TIME.strftime('%Y-%m-%d %H:%M:%S')}[/dim]",
    title="[bold]📋 Konfigurasi[/bold]",
    border_style="cyan"
))

# ── Config ────────────────────────────────────────────────────────────────────
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; OrphanChecker/1.0; +https://example.com/bot)"
}
REQUEST_TIMEOUT = 20
DELAY_BETWEEN   = 0.3
MAX_PAGES       = 10000

SKIP_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".gif", ".webp", ".svg", ".ico", ".avif", ".bmp",
    ".mp4", ".mov", ".avi", ".webm", ".mp3", ".wav", ".ogg",
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
    ".zip", ".rar", ".gz", ".tar", ".7z",
    ".css", ".js", ".json", ".xml", ".txt", ".rss", ".atom",
    ".woff", ".woff2", ".ttf", ".eot",
}

SKIP_PREFIXES = (
    "mailto:", "tel:", "javascript:", "whatsapp:", "#", "data:",
)

# ── Helpers ───────────────────────────────────────────────────────────────────
def normalize_url(url: str) -> str:
    if not isinstance(url, str):
        return ""
    url, _ = urldefrag(url)
    return url.rstrip("/").lower()

def should_skip(url: str) -> bool:
    if any(url.startswith(p) for p in SKIP_PREFIXES):
        return True
    path = urlparse(url).path.lower()
    ext  = "." + path.split(".")[-1] if "." in path.split("/")[-1] else ""
    return ext in SKIP_EXTENSIONS

def is_internal(url: str, base: str) -> bool:
    return urlparse(url).netloc == urlparse(base).netloc

def fetch_page(url: str):
    resp         = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
    final_url    = normalize_url(resp.url)
    status_code  = resp.status_code
    content_type = resp.headers.get("content-type", "")
    is_html      = "text/html" in content_type

    links = []
    title = None
    if is_html and resp.status_code == 200:
        soup      = BeautifulSoup(resp.text, "html.parser")
        title_tag = soup.find("title")
        title     = title_tag.get_text(strip=True) if title_tag else None
        for a in soup.find_all("a", href=True):
            href = a["href"].strip()
            if not href or any(href.startswith(p) for p in SKIP_PREFIXES):
                continue
            abs_url = normalize_url(urljoin(url, href))
            if is_internal(abs_url, BASE_DOMAIN) and not should_skip(abs_url):
                links.append(abs_url)

    return {
        "final_url"   : final_url,
        "status_code" : status_code,
        "content_type": content_type,
        "title"       : title,
        "links_found" : len(links),
        "is_redirect" : final_url != normalize_url(url),
    }, links

# ── Crawler ───────────────────────────────────────────────────────────────────
def crawl_website(homepage, max_pages=MAX_PAGES):
    visited        = {}
    queue          = [normalize_url(homepage)]
    queued_set     = set(queue)
    error_count    = 0
    redirect_count = 0

    console.rule("[bold magenta]🚀 STEP 1 — CRAWLING WEBSITE[/bold magenta]")

    with Progress(
        SpinnerColumn(spinner_name="dots2", style="magenta"),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(bar_width=28, style="magenta", complete_style="green"),
        TextColumn("[bold green]{task.fields[crawled]}[/bold green] crawled"),
        TextColumn("· [yellow]{task.fields[queued]}[/yellow] queued"),
        TextColumn("· [dim]{task.fields[current]}[/dim]"),
        TimeElapsedColumn(),
        console=console,
        refresh_per_second=8,
    ) as progress:

        task = progress.add_task(
            "[magenta]Crawling...",
            total=None,
            crawled=0,
            queued=len(queue),
            current="starting...",
        )

        while queue:
            current_url = queue.pop(0)
            if current_url in visited:
                continue
            if len(visited) >= max_pages:
                console.print(f"\n[yellow]⚠  Batas max_pages={max_pages} tercapai.[/yellow]")
                break

            progress.update(
                task,
                description=f"[magenta]Page [{len(visited)+1}]",
                crawled=len(visited),
                queued=len(queue),
                current=current_url[-60:] if len(current_url) > 60 else current_url,
            )

            try:
                page_data, links = fetch_page(current_url)
                page_data["url"] = current_url
                visited[current_url] = page_data

                if page_data["is_redirect"]:
                    redirect_count += 1

                new_links = 0
                for lnk in links:
                    if lnk not in visited and lnk not in queued_set:
                        queue.append(lnk)
                        queued_set.add(lnk)
                        new_links += 1

                status = page_data["status_code"]
                status_style = "[green]" if status == 200 else "[yellow]" if status in (301, 302) else "[red]"
                console.print(
                    f"  {status_style}[{status}][/]  "
                    f"[dim]{current_url[-70:]}[/dim]  "
                    f"→ [cyan]{page_data['links_found']}[/cyan] links  "
                    f"[dim]+{new_links} new[/dim]"
                )
                time.sleep(DELAY_BETWEEN)

            except requests.Timeout:
                error_count += 1
                visited[current_url] = {"url": current_url, "status_code": "TIMEOUT",
                                         "title": None, "links_found": 0, "is_redirect": False}
                console.print(f"  [red]✗ TIMEOUT[/red]  [dim]{current_url[-70:]}[/dim]")
            except requests.HTTPError as e:
                error_count += 1
                visited[current_url] = {"url": current_url, "status_code": e.response.status_code,
                                         "title": None, "links_found": 0, "is_redirect": False}
                console.print(f"  [red]✗ HTTP {e.response.status_code}[/red]  [dim]{current_url[-70:]}[/dim]")
            except Exception as e:
                error_count += 1
                visited[current_url] = {"url": current_url, "status_code": f"ERROR:{type(e).__name__}",
                                         "title": None, "links_found": 0, "is_redirect": False}
                console.print(f"  [red]✗ {type(e).__name__}[/red]  [dim]{current_url[-70:]}[/dim]")

        progress.update(task, description="[bold green]✔ Crawl selesai!", completed=1, total=1)

    return list(visited.values()), error_count, redirect_count

# ── Run crawl ─────────────────────────────────────────────────────────────────
crawled_rows, error_count, redirect_count = crawl_website(HOMEPAGE)

END_CRAWL = datetime.now()
DUR_CRAWL = (END_CRAWL - START_TIME).total_seconds()

console.print()
stats = Table(box=box.SIMPLE, show_header=False, padding=(0, 2))
stats.add_column(style="dim")
stats.add_column(style="bold white")
stats.add_row("🔗  Total URL ditemukan", f"[bold green]{len(crawled_rows)}[/bold green]")
stats.add_row("↪   Redirect",            str(redirect_count))
stats.add_row("❌  Error",               f"[red]{error_count}[/red]" if error_count else "[green]0[/green]")
stats.add_row("⏱  Durasi crawl",        f"{DUR_CRAWL:.1f} detik")
console.print(Panel(stats, title="[bold]📊 Hasil Crawl[/bold]", border_style="magenta"))

df_crawled = pd.DataFrame(crawled_rows)
df_crawled.to_csv(OUTPUT_CRAWL, index=False, encoding="utf-8-sig")
console.print(f"  [green]✔[/green] Crawl CSV tersimpan: [cyan]{OUTPUT_CRAWL}[/cyan]\n")

# Siapkan set URL untuk Cell 2
CRAWLED_URLS = set(
    normalize_url(r["url"]) for r in crawled_rows
    if r.get("status_code") == 200
)
console.print(f"  [dim]✔ CRAWLED_URLS siap: {len(CRAWLED_URLS)} URL (status 200)[/dim]")
console.print(f"\n[bold green]✔ Cell 1 selesai! Jalankan Cell 2 sekarang.[/bold green]\n")

---
## 🔎 Cell 2 — Orphan Page Detector
Jalankan cell ini setelah Cell 1 selesai. Tidak perlu input tambahan.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Orphan Page Detector                          ║
# ╚══════════════════════════════════════════════════════════╝

console.print(Panel.fit(
    "[bold red]🔎 ORPHAN PAGE DETECTOR[/bold red]\n"
    "[dim]Bandingkan Crawled URLs vs Sitemap URLs — Step 2 of 2[/dim]",
    border_style="red"
))

console.rule("[bold red]🔎 STEP 2 — ANALISIS ORPHAN PAGE[/bold red]")

# ── Normalisasi sitemap URLs ──────────────────────────────────────────────────
PAGE_TYPES   = {"page_url", "image_url", "news_url", "video_url"}
df_pages     = df_sitemap[df_sitemap["entry_type"].isin(PAGE_TYPES)].copy()
SITEMAP_URLS = set(df_pages["loc"].dropna().apply(normalize_url))

console.print(f"\n  [green]✔[/green] Sitemap URLs (page types) : [bold]{len(SITEMAP_URLS)}[/bold]")
console.print(f"  [green]✔[/green] Crawled URLs (status 200) : [bold]{len(CRAWLED_URLS)}[/bold]\n")

# ── Perbandingan ──────────────────────────────────────────────────────────────
orphan_urls    = SITEMAP_URLS - CRAWLED_URLS
reachable_urls = SITEMAP_URLS & CRAWLED_URLS
unlisted_urls  = CRAWLED_URLS - SITEMAP_URLS

console.print(f"  [bold green]✔  Terjangkau dari homepage[/bold green]               : [bold green]{len(reachable_urls)}[/bold green] URLs")
console.print(f"  [bold red]✗  Orphan (di sitemap, tidak terjangkau)[/bold red]     : [bold red]{len(orphan_urls)}[/bold red] URLs")
console.print(f"  [bold yellow]⚠  Tidak di sitemap (crawled, tapi unlisted)[/bold yellow]  : [bold yellow]{len(unlisted_urls)}[/bold yellow] URLs\n")

# ── Build orphan dataframe ────────────────────────────────────────────────────
df_orphan = df_pages[df_pages["loc"].apply(normalize_url).isin(orphan_urls)].copy()
df_orphan = df_orphan[["loc", "lastmod", "changefreq", "priority", "entry_type"]].copy()
df_orphan = df_orphan.sort_values("lastmod", ascending=False, na_position="last")
df_orphan.insert(0, "status", "ORPHAN")

# ── Ringkasan orphan ──────────────────────────────────────────────────────────
if not df_orphan.empty:
    oc = df_orphan["entry_type"].value_counts()
    mx = oc.max()
    type_table = Table(title="Orphan — per Entry Type", box=box.ROUNDED,
                       border_style="red", header_style="bold red")
    type_table.add_column("Entry Type", style="white")
    type_table.add_column("Count",      style="bold red", justify="right")
    type_table.add_column("Bar",        style="red", no_wrap=True)
    for et, cnt in oc.items():
        bar = "█" * int(cnt/mx*18) + "░" * (18 - int(cnt/mx*18))
        type_table.add_row(et, str(cnt), bar)
    console.print(type_table)

    preview = Table(title="Preview Orphan URLs (top 10 by lastmod)",
                    box=box.ROUNDED, border_style="red", header_style="bold red")
    preview.add_column("URL",      style="dim", max_width=65, no_wrap=True)
    preview.add_column("Lastmod",  style="yellow", max_width=12)
    preview.add_column("Priority", style="cyan",   justify="center")
    preview.add_column("Type",     style="white")
    for _, row in df_orphan.head(10).iterrows():
        preview.add_row(
            str(row["loc"])[:65],
            str(row["lastmod"]  or "-"),
            str(row["priority"] or "-"),
            str(row["entry_type"]),
        )
    console.print(preview)

# ── Export & Download ─────────────────────────────────────────────────────────
END_TIME  = datetime.now()
TOTAL_DUR = (END_TIME - START_TIME).total_seconds()

df_orphan.to_csv(OUTPUT_ORPHAN, index=False, encoding="utf-8-sig")

console.print()
console.rule("[bold cyan]💾 EXPORT[/bold cyan]")
console.print(f"\n  [bold green]✔[/bold green] Crawl CSV  : [cyan]{OUTPUT_CRAWL}[/cyan]  ({len(df_crawled)} baris)")
console.print(f"  [bold green]✔[/bold green] Orphan CSV : [cyan]{OUTPUT_ORPHAN}[/cyan]  ([bold red]{len(df_orphan)}[/bold red] orphan pages)\n")

colab_files.download(OUTPUT_CRAWL)
colab_files.download(OUTPUT_ORPHAN)

console.print(Panel(
    f"  Sitemap URLs      : [white]{len(SITEMAP_URLS)}[/white]\n"
    f"  Reachable         : [bold green]{len(reachable_urls)}[/bold green]\n"
    f"  Orphan Pages      : [bold red]{len(orphan_urls)}[/bold red]\n"
    f"  Not in Sitemap    : [bold yellow]{len(unlisted_urls)}[/bold yellow]\n\n"
    f"  Total Durasi      : [dim]{TOTAL_DUR:.1f} detik[/dim]",
    title="[bold]🎉 Ringkasan Akhir[/bold]",
    border_style="green"
))